In [1]:
import numpy as np
import tensorflow as tf
import random
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error
from Data.Data_Generator_ILI_HHS_TimeSplit import prepare_ILI_data
from Utils import drop_last_n_samples, splitter
import matplotlib.pyplot as plt
import pandas as pd


2025-06-01 15:15:29.504531: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-01 15:15:30.818394: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:

# — 1. Load your DataFrame (assumed to be in `df` with columns ['ds','unique_id','y']) —
# df = pd.read_csv("your_data.csv")  # example

path_to_data = "Data/ILINet.csv"
test_size = 0.2
train_df, test_df, df, data, number_of_time_series, length_time_series, forecasting_horizon, number_of_samples_in_test = prepare_ILI_data(path_to_data, test_size)


def make_windows(group, past=52, future=4):
    """Given a 1D array, return (X_windows, y_windows)."""
    vals = group['y'].values
    X, y = [], []
    for i in range(len(vals) - past - future + 1):
        X.append(vals[i : i + past])
        y.append(vals[i + past : i + past + future])
        x_total = np.array(X)
        y_total = np.array(y)
    return x_total[:x_total.shape[0]-200,:], y_total[:y_total.shape[0]-200,:], x_total[x_total.shape[0]-200:,:], y_total[y_total.shape[0]-200:,:]

# — 3. Build training arrays —
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
for _, grp in df.groupby('unique_id'):
    Xg_train, yg_train, Xg_test, yg_test = make_windows(grp, past=52, future=4)
    X_train_list.append(Xg_train)
    y_train_list.append(yg_train)
    X_test_list.append(Xg_test)
    y_test_list.append(yg_test)
    
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

# reshape for LSTM: (samples, timesteps, features)
X_train = X_train[..., np.newaxis]   # now shape = (N, 52, 1)


def compute_nnse(y_true, y_pred):
    """
    Normalized Nash–Sutcliffe Efficiency:
      NSE  = 1 - SS_res / SS_tot
      NNSE = 1 / (2 - NSE)
    """
    # Sum of squared residuals
    ss_res = np.sum((y_true - y_pred) ** 2)
    # Total sum of squares
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    # Nash–Sutcliffe Efficiency
    nse = 1 - ss_res / ss_tot
    # Normalized NSE in [0,1]
    return 1 / (2 - nse)

def run_once(seed=None):
    # 1) (Re)load and window your data here…
    #    prepare_ILI_data, make_windows, X_train, y_train, X_test, y_test as before
    
    K.clear_session()
    if seed is not None:
        np.random.seed(seed)
        tf.random.set_seed(seed)
        random.seed(seed)
    
    # — 5. Define the LSTM model —
    model = Sequential([
        LSTM(16, return_sequences=True, input_shape=(52, 1), activation='relu'),
        LSTM(16, activation='relu'),
        Dense(32, activation='relu'),
        Dense(4)            # output 4 steps ahead
    ])

    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')
    model.summary()

    # — 6. Train —
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=100,
        batch_size=128
    )

    # — predict —
    y_pred = model.predict(X_test)
    
    # — compute per-horizon MSE & MAE —
    mse_per_h = []
    nnse_per_h = []
    
    # loop over each forecast horizon
    for h in range(y_pred.shape[1]):
        y_h_true = y_test[:, h].reshape(10, 200)
        y_h_pred = y_pred[:, h].reshape(10, 200)

        # 1) mean‐squared error
        mse = np.mean((y_h_true - y_h_pred) ** 2)
        mse_per_h.append(mse)

        # 2) normalized NSE
        nnse = compute_nnse(y_h_true, y_h_pred)
        nnse_per_h.append(nnse)
        
        print(f"Horizon {h+1}:  MSE={mse:.3f}") 
        print(f"Horizon {h+1}:  NNSE={nnse:.3f}") 
    
    return np.array(mse_per_h), np.array(nnse_per_h)

# — run 5 trials —
all_mse, all_nnse = [], []

for seed in range(8):
    mse_h, nnse_h = run_once(seed=seed)   # adjust run_once to return (mse_per_h, nnse_per_h)
    all_mse .append(mse_h)
    all_nnse.append(nnse_h)

all_mse  = np.vstack(all_mse)
all_nnse = np.vstack(all_nnse)

results = pd.DataFrame({
    'horizon': [1,2,3,4],
    'MSE'     : all_mse.mean(axis=0),
    'mse_std' : all_mse.std(axis=0),
    'NNSE'    : all_nnse.mean(axis=0),
    'nnse_std': all_nnse.std(axis=0),
})

# compute the mean of each numeric column
means = results[['MSE','mse_std','NNSE','nnse_std']].mean()

# create a new row with horizon label "Avg"
avg_row = {
    'horizon': 'Avg',
    'MSE'     : means['MSE'],
    'mse_std' : means['mse_std'],
    'NNSE'    : means['NNSE'],
    'nnse_std': means['nnse_std'],
}

# append it
results = results.append(avg_row, ignore_index=True)

print(results)


/sfs/weka/scratch/jrp5td/Geoff_test_2/Data/Data_Generator_ILI_HHS_TimeSplit.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'y'] = df['y'].astype(float)


(7450, 52)
(7450, 4)
(2000, 52)
(2000, 4)
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 52, 16)            1152      
                                                                 
 lstm_1 (LSTM)               (None, 16)                2112      
                                                                 
 dense (Dense)               (None, 32)                544       
                                                                 
 dense_1 (Dense)             (None, 4)                 132       
                                                                 
Total params: 3940 (15.39 KB)
Trainable params: 3940 (15.39 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/100
59/59 [==============================] - 7s 84ms/step - loss: 4.5897 - val_loss: 4.7785
Epoch 2/100


/tmp/ipykernel_566842/1737641830.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(avg_row, ignore_index=True)


### LSTM-direct
==== AGGREGATED RESULTS ACROSS ALL FILES =====

Average MSE:         mean=0.449, std=0.050

--- H1 ---
MSE: mean=0.145, std=0.019
NNSE: mean=0.938, std=0.001

--- H2 ---
MSE: mean=0.350, std=0.038
NNSE: mean=0.867, std=0.012

--- H3 ---
MSE: mean=0.564, std=0.063
NNSE: mean=0.807, std=0.017

--- H4 ---
MSE: mean=0.735, std=0.081
NNSE: mean=0.765, std=0.014


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
import random
import pandas as pd
from tensorflow.keras import backend as K

path_to_data = "Data/ILINet.csv"
test_size = 0.2
train_df, test_df, df, data, number_of_time_series, length_time_series, forecasting_horizon, number_of_samples_in_test = prepare_ILI_data(path_to_data, test_size)


def make_windows(group, past=52, future=4):
    """Given a 1D array, return (X_windows, y_windows)."""
    vals = group['y'].values
    X, y = [], []
    for i in range(len(vals) - past - future + 1):
        X.append(vals[i : i + past])
        y.append(vals[i + past : i + past + future])
        x_total = np.array(X)
        y_total = np.array(y)
    return x_total[:x_total.shape[0]-200,:], y_total[:y_total.shape[0]-200,:], x_total[x_total.shape[0]-200:,:], y_total[y_total.shape[0]-200:,:]

# — 3. Build training arrays —
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
for _, grp in df.groupby('unique_id'):
    Xg_train, yg_train, Xg_test, yg_test = make_windows(grp, past=52, future=4)
    X_train_list.append(Xg_train)
    y_train_list.append(yg_train)
    X_test_list.append(Xg_test)
    y_test_list.append(yg_test)
    
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

# reshape for LSTM: (samples, timesteps, features)
X_train = X_train[..., np.newaxis]   # now shape = (N, 52, 1)
X_test = X_test[..., np.newaxis]

def compute_nnse(y_true, y_pred):
    """
    Normalized Nash–Sutcliffe Efficiency:
      NSE  = 1 - SS_res / SS_tot
      NNSE = 1 / (2 - NSE)
    """
    # Sum of squared residuals
    ss_res = np.sum((y_true - y_pred) ** 2)
    # Total sum of squares
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    # Nash–Sutcliffe Efficiency
    nse = 1 - ss_res / ss_tot
    # Normalized NSE in [0,1]
    return 1 / (2 - nse)


def run_once_recursive(seed=None):
    # Reset & seed
    K.clear_session()
    if seed is not None:
        np.random.seed(seed)
        tf.random.set_seed(seed)
        random.seed(seed)

    # --- 1) Build/train a model that outputs ONE step ahead ---
    model = Sequential([
        LSTM(16, return_sequences=True, input_shape=(52, 1), activation='relu'),
        LSTM(16, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)            # single‐step output
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')
    # model.summary()

    # Train on (X_train, y_train_one_step) → we need to replace your existing y_train[:, h] with just the t+1 target
    # Instead of y_train being shape (N, 4), we want y_train_one = next‐value only.
    # If your original make_windows was returning y_windows of shape (num_windows, 4),
    # you can simply take y_train_one = y_train_full[:, 0].
    y_train_one = y_train[:, 0].reshape(-1, 1)   # shape = (N, 1)
    y_test_full = y_test.copy()                 # keep the full 4‐step ground truth aside

    history = model.fit(
        X_train, y_train_one,
        validation_data=(X_test, y_test_one),   # you’ll need to define y_test_one = y_test[:, 0].reshape(-1, 1)
        epochs=100,
        batch_size=128,
        verbose=1
    )

    # --- 2) At inference, do a 4‐step loop per sample ---
    # We'll build an array preds_rec of shape (num_samples, 4)
    num_samples = X_test.shape[0]
    preds_rec = np.zeros((num_samples, 4), dtype=float)

    # For each test sample:
    for i in range(num_samples):
        # a) start with the “last 52‐point input window” for sample i
        #    X_test[i] has shape (52, 1). We flatten it to a 1D array of length 52:
        window = X_test[i, :, 0].copy()   # shape = (52,)
        
        # b) now loop h=0..3, predict one at a time:
        for h in range(4):
            # reshape window → (1, 52, 1) for model.predict
            window_reshaped = window.reshape(1, 52, 1)
            y_hat = model.predict(window_reshaped, verbose=0)  # shape = (1, 1)
            y_hat_value = float(y_hat[0, 0])
            preds_rec[i, h] = y_hat_value

            # c) “roll” the window: drop oldest and append y_hat_value
            window = np.roll(window, -1)   # shift left by 1
            window[-1] = y_hat_value       # put predicted value at the end

    # preds_rec is now shape (num_samples, 4). We can compare to y_test_full[:, 0..3].
    mse_per_h = []
    nnse_per_h = []
    for h in range(4):
        # Reshape to (10 regions, 200 timesteps) if you want to group by region–
        # exactly as you did before
        y_h_true = y_test_full[:, h].reshape(10, 200)
        y_h_pred = preds_rec[:, h].reshape(10, 200)

        mse = np.mean((y_h_true - y_h_pred) ** 2)
        nnse = compute_nnse(y_h_true, y_h_pred)

        mse_per_h.append(mse)
        nnse_per_h.append(nnse)
        print(f"Horizon {h+1}:  MSE={mse:.3f},  NNSE={nnse:.3f}")

    return np.array(mse_per_h), np.array(nnse_per_h)


# --- MAIN LOOP: run 5 trials, collect metrics ---
all_mse, all_nnse = [], []
# But first, define y_test_one = y_test[:, 0].reshape(-1, 1) once:
y_train_one = y_train[:, 0].reshape(-1, 1)
y_test_one  = y_test[:, 0].reshape(-1, 1)

for seed in range(8):
    mse_h, nnse_h = run_once_recursive(seed)
    all_mse.append(mse_h)
    all_nnse.append(nnse_h)

all_mse  = np.vstack(all_mse)   # shape = (5, 4)
all_nnse = np.vstack(all_nnse)

results = pd.DataFrame({
    'horizon': [1,2,3,4],
    'MSE'     : all_mse.mean(axis=0),
    'mse_std' : all_mse.std(axis=0),
    'NNSE'    : all_nnse.mean(axis=0),
    'nnse_std': all_nnse.std(axis=0),
})

# add an “Avg” row just as before
means = results[['MSE','mse_std','NNSE','nnse_std']].mean()
avg_row = {
    'horizon': 'Avg',
    'MSE'     : means['MSE'],
    'mse_std' : means['mse_std'],
    'NNSE'    : means['NNSE'],
    'nnse_std': means['nnse_std'],
}
results = results.append(avg_row, ignore_index=True)
print(results)


/sfs/weka/scratch/jrp5td/Geoff_test_2/Data/Data_Generator_ILI_HHS_TimeSplit.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'y'] = df['y'].astype(float)


(7450, 52)
(7450, 4)
(2000, 52)
(2000, 4)


2025-06-01 15:15:43.406255: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1636] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31013 MB memory:  -> device: 0, name: Tesla V100-SXM2-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


Epoch 1/100


2025-06-01 15:15:46.806180: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7ff11c002d10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-01 15:15:46.806242: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Tesla V100-SXM2-32GB, Compute Capability 7.0
2025-06-01 15:15:47.043312: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:255] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-06-01 15:15:47.291153: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8904
2025-06-01 15:15:48.100112: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


59/59 [==============================] - 10s 87ms/step - loss: 4.1478 - val_loss: 4.0490
Epoch 2/100
59/59 [==============================] - 5s 82ms/step - loss: 2.8213 - val_loss: 1.9691
Epoch 3/100
59/59 [==============================] - 5s 83ms/step - loss: 1.8343 - val_loss: 1.2854
Epoch 4/100
59/59 [==============================] - 5s 87ms/step - loss: 1.4079 - val_loss: 0.9678
Epoch 5/100
59/59 [==============================] - 5s 85ms/step - loss: 1.0701 - val_loss: 0.6916
Epoch 6/100
59/59 [==============================] - 5s 85ms/step - loss: 0.8063 - val_loss: 0.6250
Epoch 7/100
59/59 [==============================] - 5s 82ms/step - loss: 0.6431 - val_loss: 0.4185
Epoch 8/100
59/59 [==============================] - 5s 85ms/step - loss: 0.5383 - val_loss: 0.3423
Epoch 9/100
59/59 [==============================] - 6s 93ms/step - loss: 0.4716 - val_loss: 0.3126
Epoch 10/100
59/59 [==============================] - 5s 92ms/step - loss: 0.4164 - val_loss: 0.2887
Epoch 11/1

/tmp/ipykernel_566842/1451928321.py:172: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(avg_row, ignore_index=True)


### LSTM-iterative
==== AGGREGATED RESULTS ACROSS ALL FILES =====

Average MSE:         mean=0.478, std=0.019

--- H1 ---
MSE: mean=0.121, std=0.002
NNSE: mean=0.947, std=0.002

--- H2 ---
MSE: mean=0.358, std=0.010
NNSE: mean=0.864, std=0.003

--- H3 ---
MSE: mean=0.605, std=0.033
NNSE: mean=0.796, std=0.009

--- H4 ---
MSE: mean=0.829, std=0.045
NNSE: mean=0.740, std=0.012
